# fm-chem on ColabMinimal flow matching: standard linear interpolant, no guidance, plain ODE vector field.Two tracks, same flow matching code underneath.- **Track A — conformers.** rMD17: 100k DFT conformations of a fixed molecule. A thermalized  conformational ensemble, which is the same problem shape as an ion-pair ensemble.- **Track B — latents.** MOSES + a SELFIES VAE, then flow matching in the latent space.Set **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
!git clone https://github.com/YOURNAME/fm-chem.git 2>/dev/null || echo "or upload the folder"%cd fm-chem!pip install -q rdkit selfiesimport torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## Step 0 — verify the implementation is correctThree quantitative tests on 2-D toys. Do this first, every time you change `fm/`.

In [ ]:
!python scripts/sanity_check.py

## Track A — molecular conformers`prep_rmd17.py` downloads one molecule (~65 MB) and featurizes it.The `--featurizer` flag is the experiment worth running twice:- `aligned` — Kabsch-align to a reference frame. 3N dims, exactly invertible.- `distance` — pairwise distances. Rotation-invariant by construction, but  reconstruction is approximate and it cannot tell mirror images apart.Neither is equivariant. Comparing them shows you concretely what equivariantarchitectures would buy you.

In [ ]:
!python scripts/prep_rmd17.py --molecule ethanol --featurizer aligned!python scripts/train_vectors.py \    --x1 data/ethanol_aligned_x.npy --out runs/ethanol \    --steps 20000 --width 512 --depth 6 --batch-size 512 --n-samples 5000

In [ ]:
# The metrics the FM loss cannot give you.# Bond MEANS are easy to match. Bond STANDARD DEVIATIONS are the real test:# they say whether the model captured the width of the thermal ensemble# or collapsed onto the average structure.!python scripts/eval_conformers.py \    --samples runs/ethanol/samples.npy --meta data/ethanol_aligned_meta.pkl

### Track A experiments worth running1. **Featurization.** Rerun with `--featurizer distance`. Which gets better bond   statistics? Check how often MDS reconstruction produces large negative Gram   eigenvalues — those are geometrically impossible structures.2. **Solver budget.** Sweep `--ode-steps` over 5, 10, 20, 50, 100. A well-trained   linear-path model degrades gracefully; a badly-trained one falls apart at low   step counts. This is a sharper diagnostic than the loss curve.3. **Multimodality.** Switch to `--molecule malonaldehyde`, which has an   intramolecular proton transfer and a genuinely bimodal ensemble. Does the   model reproduce both wells, or average them into one?4. **Temperature conditioning.** rMD17 is a single temperature, so you cannot do   this directly here — but MD22 has trajectories at 400–500 K. This is the step   that connects to the ionic liquid idea.

## Track B — flow matching in a learned latent spaceTwo stages. Stage 1 trains the VAE; stage 2 runs flow matching on its latents.**Check the VAE's reconstruction accuracy before moving to stage 2.** It sets thequality ceiling for the whole track. If it cannot round-trip its own trainingdata, flow matching cannot fix that.

In [ ]:
!python scripts/prep_moses.py --n 200000!python scripts/train_latent_ae.py --epochs 15 --beta 3e-3 --latent-dim 64

In [ ]:
!python scripts/train_latent_fm.py --steps 30000 --width 512 --depth 6

### Track B experiments worth running1. **The `beta` sweep.** This is the central experiment. Try `--beta` at 0, 1e-4,   3e-3, 1e-1. At 0 you have a plain autoencoder: excellent reconstruction, and   flow matching samples that decode to garbage because the latent is full of   holes the decoder never learned. At high beta the latent is smooth but   reconstruction collapses. Plot reconstruction accuracy against sample   uniqueness/novelty and you have the whole tradeoff on one figure.2. **Latent dimension.** 16 / 64 / 256. Larger is easier to reconstruct from and   harder to model with flow matching.3. **Read the metrics in the right order.** Uniqueness catches mode collapse,   which the FM loss is blind to. Novelty catches memorization. Validity is near   100% by SELFIES construction and proves nothing.

## What neither track gives youThe ODE time `t` is the path time of the generative process, not physical time.The intermediates `x_t` are points on a straight line between a source sample anda target sample. They are not reaction coordinates, not transition states, notfolding intermediates — however suggestive they look when animated.